In [1]:
import Pkg;
Pkg.activate(@__DIR__);
Pkg.instantiate()

  Activating new environment at `c:\Users\sesch\Documents\Projects\Cart Pole\Controller\TinyMPC\Project.toml`


  No Changes to `C:\Users\sesch\Documents\Projects\Cart Pole\Controller\TinyMPC\Project.toml`
  No Changes to `C:\Users\sesch\Documents\Projects\Cart Pole\Controller\TinyMPC\Manifest.toml`


In [3]:
using LinearAlgebra
using BlockDiagonals
using ForwardDiff
using Plots
using Random
using Printf
using MeshCat
using TrajOptPlots
using RobotZoo:Cartpole
using StaticArrays
using GeometryTypes
using ColorTypes

In [4]:
vis = Visualizer()
model = Cartpole()
TrajOptPlots.set_mesh!(vis, model)

┌ Info: MeshCat server started. You can open the visualizer by visiting the following URL in your browser:
│ http://127.0.0.1:8701
└ @ MeshCat C:\Users\sesch\.julia\packages\MeshCat\GlCMx\src\visualizer.jl:73


MeshCat Visualizer with path /meshcat/robot/cart/pole at http://127.0.0.1:8701

In [25]:
# Cartpole params
m1 = 1
m2 = 1
ℓ = 0.12

g = 9.81

h = 1/500

0.002

In [30]:
# x = [x, v, θ, θdot]

function cartpole_dynamics(x,u)
  r = x[1] # cart position
  ṙ = x[2] # change in cart position
  θ = x[3] # pole angle
  θ̇ = x[4] # change in pole angle

  F = u

  ẍ = [-ℓ*(F + (θ̇ ^2)*m2*ℓ*sin(θ))/(-ℓ*(m1+m2) + m2*ℓ*cos(θ)^2) + (-g*m2*ℓ*cos(θ)*sin(θ))/(-ℓ*(m1+m2)+m2*ℓ*cos(θ)^2);
        ((F + θ̇ *m2*ℓ*sin(θ))*cos(θ))/(-ℓ*(m1+m2) + m2*ℓ*cos(θ)^2) + (-g*(-m1-m2)*sin(θ))/(-ℓ*(m1+m2) + m2*ℓ*cos(θ)^2)]

  r̈ = ẍ[1]
  θ̈ = ẍ[2]

  return [ṙ; r̈; θ̇; θ̈ ]
end
function cartpole_rk4(x,u)
  #RK4 integration with zero-order hold on u
  f1 = cartpole_dynamics(x, u)
  f2 = cartpole_dynamics(x + 0.5*h*f1, u)
  f3 = cartpole_dynamics(x + 0.5*h*f2, u)
  f4 = cartpole_dynamics(x + h*f3, u)
  xn = x + (h/6.0)*(f1 + 2*f2 + 2*f3 + f4)
  return xn
end

cartpole_rk4 (generic function with 1 method)

In [31]:
rg = 0
vg = 0
θg = pi
θ̇g = 0
xg = [rg; vg; θg; θ̇g];

In [50]:
# Linearize dynamics about vertical
A = ForwardDiff.jacobian(x->cartpole_rk4(x,0),xg);
B = ForwardDiff.derivative(u->cartpole_rk4(xg,u),0);

In [51]:
display(A)
display(B)

4×4 Matrix{Float64}:
 1.0  0.002  1.96211e-5  1.308e-8
 0.0  1.0    0.0196221   1.96211e-5
 0.0  0.0    1.00033     0.00200022
 0.0  0.0    0.327036    1.00033

4-element Vector{Float64}:
 2.0000545000000002e-6
 0.002000109
 1.6667575000000004e-5
 0.016668483333333334

In [73]:
Tfinal = 5.0            # final time
N = Int(Tfinal/h)+1     # number of time steps
t_vec = h*(0:N-1)

nx = 4
nu = 1

# Cost weights
Q = Array(Diagonal([1; 1; 1; 1]));
R = 1
Qf = 1*Q;

# Penalty
ρ = 5
R̃ = R + ρ*I;

# Precompute
cache = (
    Ã = A,
    B̃ = B,
    Kinf = zeros(nu,nx),
    Pinf = zeros(nx,nx),
    Quu_inv = zeros(nu,nu),
    AmBKt = zeros(nx,nx), 
    coeff_d2p = zeros(nx,nu), 
)

K = [zeros(nu,nx) for i = 1:N-1]; # Feedback gain
P = [zeros(nx,nx) for i = 1:N];   # Cost to go quadratic term
P[N] = Qf


for k = (N-1):-1:1
    K[k] = (R̃ + B'*P[k+1]*B)\(B'*P[k+1]*A);
    P[k] = Q + A'*P[k+1]*(A - B*K[k]);
end

cache.Kinf .= K[1];
cache.Pinf .= P[1];
cache.Quu_inv .= (R̃ + cache.B̃'*cache.Pinf*cache.B̃);
cache.AmBKt .= (cache.Ã - cache.B̃*cache.Kinf)';
cache.coeff_d2p .= cache.Kinf'*R̃ - cache.AmBKt*cache.Pinf*cache.B̃;


# Create trajectory to follow
Xref = [xg for i = 1:N]
Uref = [zeros(nu) for i = 1:N-1]


# u_min = 0.1*[1; 1; 1; 1] - uhover
# u_max =  uhover - 0.1*[1; 1; 1; 1] 
u_min = -5
u_max = 5

In [74]:
include("TinyMPC.jl")

export_mat_to_c (generic function with 1 method)